# L4: Ferramentas para uma campanha de divulgação do cliente
Nesta lição, você aprenderá mais sobre Ferramentas. Você se concentrará em três elementos principais das Ferramentas:

* Versatilidade
* Tolerância a falhas
* Cache

In [25]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [26]:
from crewai import Agent, Task, Crew
import yaml 
import os

In [27]:
with open("config.yaml", "r") as config_file:
  config = yaml.safe_load(config_file)
os.environ["OPENAI_API_KEY"] = config.get("OPENAI_API_KEY")
os.environ["SERP_API_KEY"] = config.get("SERP_API_KEY")

# Creating Agents

In [28]:
sales_rep_agent = Agent(
    role="Representante de Vendas",
    goal="Identificar leads de alto valor que correspondam "
         "ao nosso perfil ideal de cliente",
    backstory=(
        "Como parte da equipe dinâmica de vendas da CrewAI, "
        "sua missão é explorar o "
        "cenário digital em busca de potenciais leads. "
        "Munido de ferramentas de ponta "
        "e uma mentalidade estratégica, você analisa dados, "
        "tendências e interações para "
        "descobrir oportunidades que outros podem não perceber. "
        "Seu trabalho é crucial para abrir caminho "
        "para interações significativas e impulsionar o crescimento da empresa."
    ),
    allow_delegation=False,
    verbose=True
)

In [29]:
lead_sales_rep_agent = Agent(
    role="Representante Líder de Vendas",
    goal="Cultivar leads com comunicações personalizadas e atraentes",
    backstory=(
        "Dentro do vibrante ecossistema do departamento de vendas da CrewAI, "
        "você se destaca como a ponte entre clientes potenciais "
        "e as soluções de que eles precisam."
        "Ao criar mensagens envolventes e personalizadas, "
        "você não apenas informa os leads sobre nossas ofertas, "
        "mas também os faz se sentirem vistos e ouvidos."
        "Seu papel é fundamental para converter interesse "
        "em ação, guiando os leads na jornada "
        "da curiosidade ao compromisso."
    ),
    allow_delegation=False,
    verbose=True
)

# Creating Tools

crewAI Tools

In [30]:
from crewai_tools import DirectoryReadTool, \
                         FileReadTool, \
                         SerperDevTool

In [31]:
directory_read_tool = DirectoryReadTool(directory='./instructions')
file_read_tool = FileReadTool()
search_tool = SerperDevTool()

Custom Tool
* Crie uma ferramenta personalizada usando a classe BaseTool do CrewAi

In [32]:
from crewai_tools import BaseTool

* Cada ferramenta precisa ter um nome e uma descrição.
* Para simplificar e para fins de sala de aula, SentimentAnalysisTool retornará positivo para cada texto.
* Ao executar localmente, você pode personalizar o código com sua lógica na função _run.

In [33]:
class SentimentAnalysisTool(BaseTool):
    name: str ="Sentiment Analysis Tool"
    description: str = ("Analyzes the sentiment of text "
         "to ensure positive and engaging communication.")
    
    def _run(self, text: str) -> str:
        # Your custom code tool goes here
        return "positive"

In [34]:
sentiment_analysis_tool = SentimentAnalysisTool()

# Creating Tasks
* A tarefa de criação de perfil de lead usa ferramentas CrewAI.

In [35]:
lead_profiling_task = Task(
    description=(
        "Conduza uma análise aprofundada de {lead_name}, "
        "uma empresa do setor {industry} "
        "que recentemente demonstrou interesse em nossas soluções. "
        "Utilize todas as fontes de dados disponíveis "
        "para compilar um perfil detalhado, "
        "focando nos principais tomadores de decisão, "
        "desenvolvimentos recentes nos negócios e necessidades potenciais "
        "que se alinhem com nossas ofertas. "
        "Esta tarefa é crucial para adaptar "
        "nossa estratégia de engajamento de forma eficaz.\n"
        "Não faça suposições e "
        "use apenas informações sobre as quais tenha absoluta certeza."
    ),
    expected_output=(
        "Um relatório abrangente sobre {lead_name}, "
        "incluindo histórico da empresa, "
        "pessoal-chave, marcos recentes e necessidades identificadas. "
        "Destaque áreas potenciais onde "
        "nossas soluções possam agregar valor "
        "e sugira estratégias de engajamento personalizadas."
    ),
    tools=[directory_read_tool, file_read_tool, search_tool],
    agent=sales_rep_agent,
)


* A tarefa de divulgação personalizada está usando sua ferramenta personalizada SentimentAnalysisTool, bem como SerperDevTool da tripulaçãoAI (search_tool).

In [36]:
personalized_outreach_task = Task(
    description=(
        "Usando os insights obtidos a partir "
        "do relatório de perfil de leads sobre {lead_name}, "
        "crie uma campanha de contato personalizada "
        "destinada a {key_decision_maker}, "
        "{position} de {lead_name}. "
        "A campanha deve abordar o recente {milestone} deles "
        "e como nossas soluções podem apoiar seus objetivos. "
        "Sua comunicação deve ressoar "
        "com a cultura e os valores da empresa {lead_name}, "
        "demonstrando um profundo entendimento do "
        "negócio e das necessidades deles.\n"
        "Não faça suposições e use apenas "
        "informações sobre as quais tenha absoluta certeza."
    ),
    expected_output=(
        "Uma série de rascunhos de e-mails personalizados "
        "adaptados para {lead_name}, "
        "especificamente direcionados a {key_decision_maker}. "
        "Cada rascunho deve incluir "
        "uma narrativa convincente que conecte nossas soluções "
        "aos recentes resultados e objetivos futuros deles. "
        "Garanta que o tom seja envolvente, profissional "
        "e alinhado com a identidade corporativa de {lead_name}."
    ),
    tools=[sentiment_analysis_tool, search_tool],
    agent=lead_sales_rep_agent,
)

# Creating the Crew

In [37]:
crew = Crew(
    agents=[sales_rep_agent, 
            lead_sales_rep_agent],
    
    tasks=[lead_profiling_task, 
           personalized_outreach_task],
	
    verbose=2,
	memory=True
)

2025-01-23 14:21:08,696 - 29136 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed


Running the Crew

In [38]:
inputs = {
    "lead_name": "DeepLearningAI",
    "industry": "Plataforma de Aprendizado Online",
    "key_decision_maker": "Andrew Ng",
    "position": "CEO",
    "milestone": "lançamento de produto"
}

result = crew.kickoff(inputs=inputs)

 [DEBUG]: == Working Agent: Representante de Vendas
 [INFO]: == Starting Task: Conduza uma análise aprofundada de DeepLearningAI, uma empresa do setor Plataforma de Aprendizado Online que recentemente demonstrou interesse em nossas soluções. Utilize todas as fontes de dados disponíveis para compilar um perfil detalhado, focando nos principais tomadores de decisão, desenvolvimentos recentes nos negócios e necessidades potenciais que se alinhem com nossas ofertas. Esta tarefa é crucial para adaptar nossa estratégia de engajamento de forma eficaz.
Não faça suposições e use apenas informações sobre as quais tenha absoluta certeza.


> Entering new CrewAgentExecutor chain...
Primeiro, vou começar procurando informações básicas sobre a empresa DeepLearningAI. Para isso, vou usar a ferramenta de pesquisa na internet.

Action: 
Search the internet

Action Input: 
{
"search_query": "DeepLearningAI company information"
}
 

I encountered an error while trying to use the tool. This was the error:

In [39]:
from IPython.display import Markdown
Markdown(result)

Aqui estão três rascunhos de emails para a campanha de contato personalizada destinada a Andrew Ng, CEO da DeepLearningAI:

Email 1: 
"Olá Andrew, 

Espero que esteja bem. Fiquei muito impressionado com o recente lançamento de produto da DeepLearningAI. Acredito que nossas soluções podem ajudá-los a atingir seus objetivos. Vamos conversar mais sobre isso? 

Atenciosamente, 

[Seu nome]"

Email 2: 
"Caro Andrew, 

Tive a oportunidade de conhecer mais sobre o trabalho incrível que a DeepLearningAI está fazendo. Acredito que podemos colaborar para impulsionar ainda mais seus esforços. Que tal marcarmos uma reunião para discutir como nossas soluções podem se alinhar com seus objetivos? 

Atenciosamente, 

[Seu nome]"

Email 3: 
"Oi Andrew, 

Estou realmente animado com o que a DeepLearningAI está fazendo e acredito que temos uma oportunidade incrível de colaborar. Que tal marcarmos um bate-papo para explorar como nossas soluções podem ajudar a DeepLearningAI a ir ainda mais longe? 

Aguardo sua resposta, 

[Seu nome]"